In [1]:
%run Latex_macros.ipynb

<IPython.core.display.Latex object>

# Fine-tuning approaches: one map

## Where this session fits

> **Every fine-tuning method answers one question: which part of a pre-trained model do we change to make it solve a new task?**

- **The question:** how do eight approaches differ, and how do we choose among them?
    - supervised fine-tuning (SFT), adapters, LoRA, BitFit
    - proxy tuning, in-context learning, hard prompts, soft prompts
- **Position in the Advanced course**
    - this session is a **prerequisite** for the reinforcement fine-tuning (RFT) sessions
    - in this course, reinforcement fine-tuning starts from an SFT model, so SFT comes first
    - the SFT practice assignment applies the first ideas: a classifier head and LoRA
- **Series goal:** adapt a pre-trained language model to a task, and know the options well enough to choose

**By the end you can**

1. say, for each of eight approaches, what is changed and what stays frozen
2. sketch the mechanism of each one
3. read the comparison tables: cost, storage, access needed, main limitation
4. place any approach on the map of four places (every weight, inside the blocks, the output, the input)
5. choose an approach from what you have: data, compute, and access to the model

**Papers used:** the papers cited on each slide and listed on the last slide.


# The setting, and five words we need

**Why this slide?** All eight approaches use the same words; define them once.

1. **Base model:** the pre-trained language model. Its **weights** are its learned parameters.
2. **Target task:** the new task, given by example pairs (input $\x$, target $\y$) and a metric that scores answers.
3. **Frozen:** not changed by the adaptation. **Trained:** changed by gradient descent on target examples.
    - the hard-prompt approach below uses a search instead of gradients
4. **Logits:** the model's scores for each possible next token, before the softmax.
5. **Prompt:** the text (or vectors) that the model reads before it continues. It is the model's input, not a weight.

> **The goal: a model that predicts $\y$ from $\x$ well on the target task, while changing as little as is needed.**

- Changing little is attractive for three reasons
    - **cost:** fewer numbers to train and to store
    - **sharing:** one base model serves many tasks
    - **forgetting:** less change to what the base model already knows


# The map: where can the knob sit?

> **A language model is a pipeline: input, embedding, blocks, logits, output. Each approach puts the adjustable part somewhere on it.**

<img src="images/ftmap_map.png" width=82%>

- **We visit four places, in this order** (the figure runs left to right, input to output)
    - 1. every weight: SFT
    - 2. inside the blocks: adapters, LoRA, BitFit
    - 3. the output: proxy tuning
    - 4. the input: in-context learning, hard prompts, soft prompts
- **Color code in the mechanism pictures** (the map itself is neutral)
    - gray is frozen, orange is trained, blue is the text we supply, green is the result


# Place 1 of 4: every weight

## 1. Full supervised fine-tuning (SFT)

> **Train every weight on target-task examples. This is the baseline that the parameter-efficient methods try to cheapen.**

<img src="images/ftmap_sft.png" width=82%>

- **Cost:** gradients and optimizer state for every parameter; the whole model is stored per task
- **Strength:** the most freedom: no limit on how the weights change
- **Weakness:** the most change, so the most risk of forgetting (LoRA, below, forgets less)
- **Practice:** the SFT practice assignment trains a classifier head, then LoRA


# Place 2 of 4: inside the blocks

## 2. Adapters: small modules inside each block

> **Full SFT is costly, so add a small bottleneck module in each block and train only the modules. The base weights stay frozen.**

**Two sizes:** $d$ is the width of the vector that each block passes along (GPT-3: $12{,}288$); $a < d$ is the width of the adapter's bottleneck.

<img src="images/ftmap_adapters.png" width=52%>

- **Wiring:** "inside the blocks" says where the change sits, not how it is wired
    - **series:** the adapter follows a layer and passes its output on (Houlsby)
    - **parallel:** the module runs beside a layer and its output is added (LoRA, next slide)
- **Size, roughly:** about $d \cdot a$ numbers per adapter against about $d^2$ per layer, a fraction near $a/d$
    - an order-of-magnitude estimate from our notebook, not a number from the paper
    - example: $d = 12{,}288$ (GPT-3) and $a = 12$ give about $0.1\%$
- **Start:** the skip connection and a zero start make the adapted model equal the base model at step 0
- **Where they matter:** in the paper's study, removing the upper-block adapters (without retraining) hurt most
- **Cost at inference:** adapters placed in series add a layer, which slows inference
- **Paper:** Houlsby et al., *Parameter-Efficient Transfer Learning for NLP* (arXiv 1902.00751)


# Place 2 of 4: inside the blocks

## 3. LoRA: a low-rank change to a weight matrix

> **LoRA is a parallel adapter without the non-linearity: the change to a weight matrix is a product of two thin matrices, $\Delta \W = A B$, and only those are trained.**

<img src="images/ftmap_lora.png" width=66%>

- **Shapes:** $\W_0$ is $d \times d$ (the same $d$), $A$ is $d \times r$, $B$ is $r \times d$
    - the rank $r \ll d$
- **Why low rank:** fine-tuning appears to need few directions (intrinsic dimensionality, arXiv 2012.13255)
    - a typical finding, not a law: the next slide has a counter-example
- **Parameters per matrix:** $2dr$ instead of $d^2$
- **Start:** $B = 0$, so $\W_0 + AB = \W_0$ before training
- **After training:** merge $\W_0 + AB$ into one matrix, so inference costs nothing extra

**LoRA: what the paper reports, and what it costs**

- **The LoRA paper's result** (GPT-3)
    - 37.7 million trained parameters, about $0.02\%$ of GPT-3, matched or beat full fine-tuning
    - it adapts only some of the matrices (the attention projections)
    - in its rank experiment, $r = 2$ scored almost as well as $r = 64$
- **A different study: "LoRA learns less and forgets less"** (Biderman et al., arXiv 2405.09673)
    - on some tasks the needed change is high rank, and LoRA under-performs full SFT
    - the rank that suffices depends on the task
    - there, the rank needed was 10 to 100 times the typical small $r$
    - its weights stay closer to the base model, so it forgets less of the source task
- **Paper:** Hu et al., *LoRA: Low-Rank Adaptation of Large Language Models* (arXiv 2106.09685)


# Place 2 of 4: inside the blocks

## 4. BitFit: train only the bias terms

> **Freeze every weight matrix and train only the bias terms $b$.**

<img src="images/ftmap_bitfit.png" width=74%>

- **Cost:** the bias terms are a tiny fraction of the parameters
- **The paper's finding** (on masked language models)
    - small and medium target datasets: comparable to training everything
    - large datasets: comparable to other sparse methods
- **Paper:** Ben Zaken et al., *BitFit* (arXiv 2106.10199)


# Place 3 of 4: the output

## 5. Proxy tuning: tune a small model, steer a large one

> **Fine-tune a small model, then add the change in its logits to the large model's logits. The large model is never trained.**

<img src="images/ftmap_proxy.png" width=54%>

- **Motive:** we cannot, or do not want to, train the large model
- **Notation**
    - $s(M, x)$: the logits of model $M$ on input $x$, the prompt so far
    - $M$: the large model, and likewise for the small models
    - $\tilde M^{FT}$: the steered model that we use
- **Recipe:** $s(\tilde M^{FT}, x) = s(M, x) + \bigl[s(M_{small}^{FT}, x) - s(M_{small}, x)\bigr]$, then softmax
- **Cost:** the expense of fine-tuning the small model only
- **Needs (our reading of the recipe):** the large model's logits, and a small model with the same vocabulary

**Proxy tuning: what the paper reports, and the caveats**

- **The paper's result:** Llama-2-70B steered by a fine-tuned Llama-2-7B recovers about $88\%$ of the gain of truly fine-tuning the 70B model
    - the task: turning the base model into a chat assistant
    - the figure is an average over tasks
- **Caveats (our reading, not the paper's)**
    - three models run at inference: the large model and both small models
    - the gain depends on the small model learning the behavior that we want
- **Paper:** Liu et al., *Tuning Language Models by Proxy* (arXiv 2401.08565)


# Place 4 of 4: the input

## 6. In-context learning: examples in the prompt

> **Put a few worked examples (exemplars) in the prompt and let the model continue the pattern. Nothing is trained.**

<img src="images/ftmap_icl.png" width=72%>

- **Formally:** the model computes $p(\y \mid C, \x)$ for a context $C$ of $k$ exemplars, instead of $p(\y \mid \x)$
- **Strength:** no training, no access to the weights; works through an API
- **Cost:** the exemplars take up the context on every call, and the task is re-described each time
- **Sensitive to** the choice and the format of the exemplars
- **Why it works is open:** see the *In-Context Learning: Theory* notebook and Min et al. (arXiv 2202.12837)
- **Origin:** Brown et al., *Language Models are Few-Shot Learners* (arXiv 2005.14165)


# Place 4 of 4: the input

## 7. Hard prompt tuning: search for the best prefix

> **In-context learning depends on the exact words, so treat the prompt as something to optimize: choose the prefix tokens that maximize a metric.**

<img src="images/ftmap_hard.png" width=74%>

- **Prefix:** tokens placed before the input
- **Scored on** a held-out tuning set, kept apart from the test set
- **Why search:** tokens are discrete, so we cannot differentiate with respect to them
- **Result:** a prefix a human can read; nothing is added to the model
- **Cost:** many forward passes to score candidate prefixes
- **Contrast:** prompt *engineering* is the same goal done by hand and by rules of thumb


# Place 4 of 4: the input

## 8. Soft prompt and prefix tuning: learn the prefix as vectors

> **Replace the prefix tokens by $m$ trainable embedding vectors and train them by gradient descent through the frozen model.**

<img src="images/ftmap_soft.png" width=64%>

- **What makes it trainable:** the first layer turns tokens into continuous vectors, so we can skip the tokens and learn the vectors
- **Trained numbers:** $m \times d$ per task for soft prompts; the model is untouched
- **Prefix tuning** adds such vectors at every block, so it has about $m \times d$ per block and sits between places 2 and 4
- **Papers**
    - Lester et al., *The Power of Scale for Parameter-Efficient Prompt Tuning* (arXiv 2104.08691)
    - Li and Liang, *Prefix-Tuning* (arXiv 2101.00190)
- **Reported results**
    - Lester et al.: the gap to full fine-tuning closes as the base model grows
    - Li and Liang (GPT-2): prefix tuning similar to or better than full fine-tuning, with about $0.1\%$ of the parameters


# Comparison 1 of 2: what changes, and what it costs

> **Four columns separate the approaches: what is changed, how much is trained, what is stored per task, and whether the base weights move.**

| Approach | What is changed | Trained numbers | Stored per task | Base weights changed? |
|:--|:--|:--|:--|:--|
| Full SFT | every weight | all | a full model copy | yes, all |
| Adapters | small modules in the blocks | roughly $a/d$ of the model | the modules | no |
| LoRA | low-rank $\Delta \W = AB$ | $2dr$ per matrix | $A$ and $B$ | no (can be merged) |
| BitFit | the bias terms | a tiny fraction | the biases | only the biases |
| Proxy tuning | the output logits | the small model, all of it | the tuned small model (the base small model is shared) | no |
| In-context learning | the prompt text | none | the exemplars | no |
| Hard prompt tuning | prefix tokens (searched) | none (a search) | the prefix tokens | no |
| Soft prompt | prefix vectors | $m \times d$ | the vectors | no |
| Prefix tuning | vectors at every block | about $m \times d$ per block | the vectors | no |


# Comparison 2 of 2: what each needs, and where it breaks

> **The second set of columns decides which approach is possible for you: access to the model comes before everything else.**

| Approach | Needs from the large model | Main limitation |
|:--|:--|:--|
| Full SFT | weights and gradients | cost; most forgetting |
| Adapters | weights and gradients | series adapters slow inference |
| LoRA | weights and gradients | under-performs when the needed change is high rank |
| BitFit | weights and gradients | paper tested on masked language models |
| Proxy tuning | its output logits only | three models run; shared vocabulary |
| In-context learning | text in, text out (an API) | exemplars fill the context window; sensitive to them |
| Hard prompt tuning | text in, text out, and a score on the task metric | a costly search over prefixes |
| Soft and prefix tuning | weights and gradients | vectors are unreadable; trails full tuning on small models (Lester et al.); prefix tuning did well on GPT-2 |


# The common structure: one picture of the differences

> **All eight aim at the same thing: a base model that does the target task well. They differ in how many numbers are learned and what access to the model is needed.**

<img src="images/ftmap_axes.png" width=58%>

- **Colors**
    - orange: trained by gradient descent
    - blue: only the input text is chosen; the model is not trained
- **Reading it**
    - moving right: more numbers are trained (not that results are better)
    - moving up: more access to the model is needed
- **Proxy tuning is hollow:** what is trained is the small model, not the large one
- **An ordering, not measurements**


# Choosing: start from what you have

> **First the access you have; then the data and memory you have; then check on held-out data.** The advice below is our rule of thumb, not a result from a paper.

1. **Access decides what is possible**
    - only text in, text out (an API): in-context learning, or a searched hard prompt
    - logits of the large model but no weights: proxy tuning, with a small model you can train
    - weights and gradients: any of the weight and prompt methods

**With weights and gradients: a default and when to leave it**

2. **Very few examples?** Start with in-context learning; it costs no training.
3. **Default for most cases: LoRA.**
    - it trains few numbers, can be merged (no extra inference cost), and one base model serves many tasks
    - adapters are similar; series adapters add inference cost
4. **Go smaller** only if memory or storage forces it: BitFit or soft prompts.
    - expect less freedom, and soft prompts trail on small base models
5. **Go bigger** if a held-out check shows LoRA is not enough: raise $r$, or use full SFT.
    - full SFT costs the most memory and risks the most forgetting

- **Always:** compare on held-out target data, and check a source-task test for forgetting


# What we established, and what it sets up

- **Established**
    - one question organizes the field: where on the frozen model does the knob sit?
    - four places: every weight, inside the blocks, the output, the input
    - eight approaches, each with trained numbers, a storage need, an access need and a limit
    - numbers marked as a paper's are that paper's own, from its own setting
    - the 0.1% adapter figure is our notebook's estimate
    - LoRA's 0.02% is a paper's result; they are different quantities
- **Sets up**
    - **in this course, reinforcement fine-tuning begins from an SFT model**
    - the same placement choices (full weights or LoRA, say) reappear when the training signal is a reward
    - the next sessions change **what the signal is**, not where the knob sits
- **Papers**
    - Adapters: arXiv 1902.00751; LoRA: arXiv 2106.09685; LoRA Learns Less and Forgets Less: arXiv 2405.09673
    - BitFit: arXiv 2106.10199; Tuning by Proxy: arXiv 2401.08565
    - In-context learning: arXiv 2005.14165 (origin), 2202.12837 (what makes it work)
    - Prompt tuning: arXiv 2104.08691; Prefix-Tuning: arXiv 2101.00190; intrinsic dimensionality: arXiv 2012.13255
